# HCAD Homestead Exemption Evidence, 2021–2025
The 2021 probable single-family rental definition used the absence of homestead evidence as one indicator that a single-family property might be renter-occupied. This notebook extends that procedure through 2025 using HCAD jurisdiction exemption records.
jur_exempt.txt contains exemption records by property account, taxing district, and exemption category. Because the same exemption can appear multiple times for one account across different taxing jurisdictions, raw rows cannot be merged directly into the property-level dataset.
Residential Homestead (RES), Partial Residential Homestead (PAR), and Apportioned Partial Residential (APR) are treated as evidence of a residential homestead exemption. The resulting dataset contains one Boolean any_homestead_evidence indicator per HCAD account appearing in the exemption file.

**Codes**\
RES	= Residential Homestead\
PAR	= Partial Residential Homestead\
APR	= Apportioned Partial Residential\
OVR	= Over-65\
DIS	= Disability\
LIH	= Low Income Housing\
SOL	= Solar\
TOT	= Total exemption\
V14	= Veteran Disability #1, 70–100%



In [1]:
from pathlib import Path
import pandas as pd
import gc

In [2]:
project = Path(r"C:\Users\alys_\Documents\Research\Evictions")
hcad_root = project / "01_DATA_RAW" / "HCAD"
output_root = project / "02_DATA_PROCESSED" / "homestead"

output_root.mkdir(parents=True, exist_ok=True)

In [3]:
years = [2021, 2022, 2023, 2024, 2025]

homestead_codes = ["RES", "PAR", "APR"]

In [4]:
def load_exempt(year):
    path = hcad_root / f"pdata{year}" / "jur_exempt.txt"

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        encoding="latin1",
        usecols=["acct", "tax_district", "exempt_cat"],
        low_memory=False
    )

    for col in ["acct", "tax_district", "exempt_cat"]:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    return df

In [5]:
def make_homestead(year):
    df = load_exempt(year)

    raw_rows = len(df)
    raw_accounts = df["acct"].nunique()
    missing_acct = df["acct"].isna().sum()
    missing_code = df["exempt_cat"].isna().sum()

    df = df[df["acct"].notna()].copy()

    df["homestead"] = df["exempt_cat"].isin(
        homestead_codes
    )

    out = (
        df.groupby("acct", as_index=False)
        .agg(
            any_homestead_evidence=(
                "homestead",
                "max"
            ),
            n_exemption_records=(
                "exempt_cat",
                "size"
            )
        )
    )

    homestead_accounts = (
        out["any_homestead_evidence"].sum()
    )

    summary = {
        "year": year,
        "raw_rows": raw_rows,
        "raw_accounts": raw_accounts,
        "missing_acct": missing_acct,
        "missing_code": missing_code,
        "final_accounts": len(out),
        "lost_accounts": raw_accounts - len(out),
        "homestead_accounts": homestead_accounts,
        "homestead_pct": round(
            homestead_accounts / len(out) * 100,
            2
        )
    }

    out.to_parquet(
        output_root / f"homestead_{year}.parquet",
        index=False
    )

    return summary

In [8]:
summary = []

for year in years:
    print("processing", year)

    s = make_homestead(year)

    summary.append(s)

    gc.collect()

processing 2021
processing 2022
processing 2023
processing 2024
processing 2025


In [9]:
df_summary_homestead = pd.DataFrame(summary)

display(df_summary_homestead)

,year,raw_rows,raw_accounts,missing_acct,missing_code,final_accounts,lost_accounts,homestead_accounts,homestead_pct
0,2021,13439842,1507911,0,0,1507911,0,824769,54.70
1,2022,13695176,1531431,0,0,1531431,0,852645,55.68
2,2023,13999771,1555713,0,0,1555713,0,868465,55.82
3,2024,14281917,1582006,0,0,1582006,0,878810,55.55
4,2025,14486769,1601909,0,0,1601909,0,881711,55.04


In [10]:
df_homestead_2022 = pd.read_parquet(
    output_root / "homestead_2022.parquet"
)

display(df_homestead_2022.head())

display(
    df_homestead_2022[
        "any_homestead_evidence"
    ].value_counts()
)

display(
    df_homestead_2022[
        "n_exemption_records"
    ].describe()
)

,acct,any_homestead_evidence,n_exemption_records
0,0010010000013,False,11
1,0010020000001,False,11
2,0010020000003,False,11
3,0010020000004,False,11
4,0010020000013,False,11


any_homestead_evidence
True     852645
False    678786
Name: count, dtype: int64

count    1531431.0
mean      8.942731
std       1.768921
min            1.0
25%            8.0
50%            9.0
75%            9.0
max           50.0
Name: n_exemption_records, dtype: Float64

## Results and decisions
The HCAD jurisdiction-exemption records were processed consistently from 2021 through 2025. The number of accounts represented increased from 1,507,911 in 2021 to 1,601,909 in 2025, and no accounts were lost during processing.
Approximately 55% of accounts showed evidence of a residential homestead exemption in each year. Residential Homestead (RES), Partial Residential Homestead (PAR), and Apportioned Partial Residential (APR) were treated as homestead evidence. Other exemption categories were not interpreted as proof of owner occupancy.
Individual accounts appeared multiple times in the raw exemption file because exemptions are recorded across taxing jurisdictions. In 2022, for example, accounts had an average of approximately 8.9 exemption records. These repeated rows were therefore collapsed to a single account-level Boolean indicator, any_homestead_evidence.
Absence of a residential homestead exemption is not interpreted by itself as proof that a property is rented. It is used only as one component of the probable single-family rental heuristic together with property class, presence of a residential building, and a mailing address different from the property address.